# Curadoria SOTA - CelebA

Filtragem baseada em:
- Similaridade semântica (CLIP)
- Consistência de pose
- Remoção de outliers

Saída: lista de imagens filtradas

In [1]:
# Instalar o CLIP oficial da OpenAI (necessário apenas uma vez)
!pip install git+https://github.com/openai/CLIP.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.7 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have num

In [2]:
import os
import torch
import clip
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 112MiB/s]


In [3]:
# Caminho dataset (já configurado)
DATASET_PATH = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"

image_paths = [
    os.path.join(DATASET_PATH, img)
    for img in os.listdir(DATASET_PATH)
    if img.endswith(".jpg")
]

print(f"Total imagens: {len(image_paths)}")

Total imagens: 202599


In [4]:
# Ajuste aqui
MAX_IMAGES = 200000

image_paths = image_paths[:MAX_IMAGES]

print(f"Usando {len(image_paths)} imagens")

Usando 200000 imagens


In [5]:
def extract_embeddings(paths, batch_size=128):
    embeddings = []
    valid_paths = []

    with torch.no_grad():
        for i in tqdm(range(0, len(paths), batch_size)):
            batch_paths = paths[i:i+batch_size]
            images = []

            for p in batch_paths:
                try:
                    img = preprocess(Image.open(p).convert("RGB"))
                    images.append(img)
                    valid_paths.append(p)
                except:
                    continue

            if len(images) == 0:
                continue

            images = torch.stack(images).to(device)

            emb = model.encode_image(images)
            emb = emb / emb.norm(dim=-1, keepdim=True)

            embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings), valid_paths

In [6]:
embeddings, image_paths = extract_embeddings(image_paths, batch_size=128)

print("Embeddings shape:", embeddings.shape)

100%|██████████| 1563/1563 [30:47<00:00,  1.18s/it] 

Embeddings shape: (200000, 512)


In [7]:
# Remoção de outliers via kNN

k = 10
nbrs = NearestNeighbors(n_neighbors=k, metric='cosine').fit(embeddings)
distances, _ = nbrs.kneighbors(embeddings)

# média da distância para vizinhos
mean_dist = distances.mean(axis=1)

# threshold adaptativo
threshold = np.percentile(mean_dist, 90)

mask = mean_dist < threshold

filtered_embeddings = embeddings[mask]
filtered_paths = np.array(image_paths)[mask]

print(f"Após remover outliers: {len(filtered_paths)} imagens")

Após remover outliers: 180000 imagens


In [8]:
# Filtragem por consistência de pose (centro do cluster)

center = filtered_embeddings.mean(axis=0)

# distância ao centro
dist_to_center = np.dot(filtered_embeddings, center)

# manter apenas mais próximos
pose_threshold = np.percentile(dist_to_center, 50)

pose_mask = dist_to_center > pose_threshold

final_paths = filtered_paths[pose_mask]

print(f"Após filtro de pose: {len(final_paths)} imagens")

Após filtro de pose: 89693 imagens


In [9]:
# Salvar lista final

OUTPUT_FILE = "filtered_celeba.txt"

with open(OUTPUT_FILE, "w") as f:
    for path in final_paths:
        f.write(path + "\n")

print("Lista salva em:", OUTPUT_FILE)

Lista salva em: filtered_celeba.txt
